# Fink/LSST — Extended Search for Cepheids in All Fields

## Why the first search found nothing

The DDFs (COSMOS, ELAIS-S1, XMM-LSS, ECDFS, EDFS, M49) are **extragalactic fields**.
Classical Cepheids in external galaxies are typically faint (m > 25) and
the LSST/Fink pipeline crossmatch radius (~1") is too small to catch them via SIMBAD
(which knows mostly MW Cepheids).  
**Milky Way Cepheids are found in galactic-plane fields**, not extragalactic DDFs.

## Extended strategy

1. **Add galactic SV fields** (Carina, Trifid-Lagoon) where MW Cepheids should be abundant
2. **Use the `cataloged` tag** via `/api/v1/tags` to fetch all cross-catalogued objects
3. **Inspect the schema** (`/api/v1/schema`) to know exactly which columns are available
4. **Explore all available crossmatch columns** from the real API response before filtering
5. **Broaden the variable-star filter**: include all pulsating/periodic variables
   (RR Lyrae, delta Scuti, W Vir, Mira) — anything on the instability strip
6. **Use `f:xm_gcvs_type` and `f:xm_vsx_Type`** with the full list of pulsating variable codes
7. **Fallback**: use Fink `resolver` to search by name (known GCVS Cepheids within the footprint)

## API notes (from swagger.json)

- `/api/v1/tags?tag=cataloged` → returns recently-catalogued objects (all types)
- `/api/v1/schema?endpoint=/api/v1/conesearch` → returns the full column schema
- `/api/v1/resolver?resolver=simbad&name_or_id=...` → resolve a known name → diaObjectId
- conesearch max radius = **18,000 arcsec (5 deg)** → use 3 deg for galactic fields

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- created : 2026-06-15
- last update : 2026-06-18 : correct for LombScargle Multiband and take scienceFlux
- last update : 2026-06-23 : check and add comments

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from astropy.timeseries import LombScargle, LombScargleMultiband
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas  {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
# to enlarge the sizes
params = {
    "legend.fontsize": "x-large",
    "figure.figsize": (10, 6),
    "axes.labelsize": "x-large",
    "axes.titlesize": "x-large",
    "xtick.labelsize": "x-large",
    "ytick.labelsize": "x-large",
}
plt.rcParams.update(params)

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("no ipympl → %matplotlib inline")

In [ ]:
# ── Fink API ──────────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org"

# ── Search parameters ─────────────────────────────────────────────────────────
NSRC_MIN = 250  # Minimum nDiaSources — cuts objects with too few detections
CONE_RADIUS = 3600.0  # Cone search radius in arcsec (1 deg per DDF)
N_ALERTS_MAX = 10000  # Max alerts per cone search
SNR_MIN = 3.0
BANDS = list("ugrizy")

# Cone radii (arcsec): DDFs use 0.5 deg, galactic fields much larger
CONE_RADIUS_DDF = 3600.0  # 1. deg for extragalactic DDFs
CONE_RADIUS_GAL = 10800.0  # 3.0 deg for galactic fields (max=18000=5deg)

# ── Lomb-Scargle period search ─────────────────────────────────────────────────
PERIOD_MIN_DAYS = 0.3
PERIOD_MAX_DAYS = 200.0  # Extend to 200 days to catch long-period Cepheids
LS_SAMPLES = 20  # Oversampling

# ── All LSST fields observed so far (DDFs + galactic SV fields) ───────────────
# Galactic fields are where we expect to find MW Cepheids!
ALL_FIELDS = {
    # ── LSST DDFs (extragalactic) ─────────────────────────────────────────────
    "COSMOS": (150.1191, 2.2058, CONE_RADIUS_DDF, "extragalactic"),
    "ELAIS-S1": (9.4500, -44.000, CONE_RADIUS_DDF, "extragalactic"),
    # "XMM-LSS": (35.7080, -4.750, CONE_RADIUS_DDF, "extragalactic"),
    "ECDFS": (53.1250, -27.800, CONE_RADIUS_DDF, "extragalactic"),
    "EDFS": (61.2400, -48.423, CONE_RADIUS_DDF, "extragalactic"),
    "M49": (187.4000, 8.000, CONE_RADIUS_DDF, "extragalactic"),
    # ── LSST SV galactic / nebular fields (where MW Cepheids live!) ───────────
    # "Carina": (161.500, -59.700, CONE_RADIUS_GAL, "galactic"),
    # "Trifid-Lagoon": (270.500, -23.000, CONE_RADIUS_GAL, "galactic"),
    # ── Additional LSST SV test fields ───────────────────────────────────────
    # "Rubin_SV_280_-48": (280.0, -48.0, CONE_RADIUS_DDF, "sv"),
    # "Rubin_SV_320_-15": (320.0, -15.0, CONE_RADIUS_DDF, "sv"),
    # "Rubin_SV_225_-40": (225.0, -40.0, CONE_RADIUS_DDF, "sv"),
}

# ── Pulsating variable star types from VSX/GCVS ───────────────────────────────
# Full list covering the classical instability strip and related pulsators
PULSATING_VSX_TYPES = {
    # Classical Cepheids (Population I)
    "CEP",
    "CEP(B)",
    "DCEP",
    "DCEPS",
    "DCEP(B)",
    # Type II Cepheids (Population II)
    "CW",
    "CWA",
    "CWB",
    # Anomalous Cepheids
    "ACEP",
    "ACEP(B)",
    # RR Lyrae (horizontal branch pulsators — same instability strip)
    "RRAB",
    "RRC",
    "RRD",
    "RR",
    # Delta Scuti / SX Phe
    "DSCT",
    "DSCT(B)",
    "SXPHE",
    "SXPHE(B)",
    # Mira / semi-regular / RV Tauri
    "M",
    "SR",
    "SRA",
    "SRB",
    "SRC",
    "SRD",
    "RVA",
    "RVB",
    "RV",
    # beta Cephei (hot pulsators)
    "BCEP",
    "BCEP(B)",
    # BL Boo (unusual Cepheid-like Population II)
    "BLBOO",
    # W Virginis
    "WVir",
}

# ── SIMBAD otype strings for pulsating stars ──────────────────────────────────
PULSATING_SIMBAD_OTYPES = {
    # Cepheids
    "Cep",
    "deltaCep",
    "WVir",
    "bCep",
    # RR Lyrae
    "RRLyr",
    # Mira / LPV
    "Mira",
    "LPV*",
    # Semi-regular
    "SRS",
    # Delta Scuti
    "dS*",
    "SX*",
    # RV Tauri
    "RV*",
    # Generic variable
    "V*",
}

# ── Strict Cepheid-only filter (for final selection) ──────────────────────────
CEPHEID_SIMBAD_STRICT = {"Cep", "deltaCep", "WVir", "bCep"}
CEPHEID_VSX_STRICT = {"CEP", "CEP(B)", "DCEP", "DCEPS", "CW", "CWA", "CWB", "ACEP", "BLBOO"}
CEPHEID_CDS_NAMES = {"Cepheid", "Classical Cepheid", "Type II Cepheid", "deltaCep", "WVir", "Cep", "bCep"}

# ── Output directories ─────────────────────────────────────────────────────────
NB_TAG = "CEPHEIDS_DDF_02"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

# ── Plot style ─────────────────────────────────────────────────────────────────
BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 12,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. API wrappers

In [ ]:
# ── Dipole columns ────────────────────────────────────────────────────────────
DIPOLE_COLS = [
    "r:isDipole",
    "r:isNegative",
    "r:dipoleFitAttempted",
    "r:dipoleFluxDiff",
    "r:dipoleFluxDiffErr",
    "r:dipoleMeanFlux",
    "r:dipoleMeanFluxErr",
    "r:dipoleLength",
    "r:dipoleAngle",
    "r:dipoleNdata",
    "r:dipoleChi2",
]

# ── Flux and source columns ───────────────────────────────────────────────────
FLUX_COLS = [
    "r:ra",
    "r:dec",
    "r:midpointMjdTai",
    "r:band",
    "r:diaObjectId",
    "r:diaSourceId",
    "r:psfFlux",
    "r:psfFluxErr",
    "r:scienceFlux",
    "r:scienceFluxErr",
    "r:templateFlux",
    "r:templateFluxErr",
    "r:apFlux",
    "r:apFluxErr",
    "r:nDiaSources",
    "r:extendedness",
    "r:reliability",
    "r:visit",
    "r:detector",
    "r:x",
    "r:y",
]

# ── Crossmatch columns ────────────────────────────────────────────────────────
CROSSMATCH_COLS = [
    "f:xm_gaiadr3_DR3Name",
    "f:xm_gaiadr3_VarFlag",
    "f:xm_gaiadr3_Plx",
    "f:xm_gaiadr3_e_Plx",
    "f:xm_gaiadr3_PhotGMag",
    "f:xm_simbad_otype",
    "f:xm_legacydr8_pstar",
    "f:xm_legacydr8_zphot",
    "f:xm_legacydr8_fqual",
    "f:xm_tns_fullname",
    "f:xm_tns_type",
    "f:xm_vsx_Type",
    "f:xm_gcvs_type",
    "f:xm_mangrove_2MASS_name",
    "f:xm_mangrove_HyperLEDA_name",
    "f:clf_cats_class",
    "f:clf_cats_score",
    "f:clf_snnSnVsOthers_score",
    "f:is_sso",
]

# ── Full column string for the API call ───────────────────────────────────────
ALL_COLS = FLUX_COLS + DIPOLE_COLS + CROSSMATCH_COLS
COLUMNS_STR = ",".join(ALL_COLS)

# Compact payload for date-window requests (avoids 400 on large fields like COSMOS)
COLUMNS_SLIM = ",".join(FLUX_COLS + DIPOLE_COLS)

print(f"Total columns requested: {len(ALL_COLS)}")
print(f"  Flux+position : {len(FLUX_COLS)}")
print(f"  Dipole        : {len(DIPOLE_COLS)}")
print(f"  Crossmatch    : {len(CROSSMATCH_COLS)}")

In [ ]:
def _post_json(url: str, payload: dict, timeout: int = 120) -> list | dict:
    """POST JSON and return parsed response; include response body for HTTP errors."""
    r = requests.post(url, json=payload, timeout=timeout)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        body = (r.text or "").strip()
        msg = f"{e}"
        if body:
            msg += f" | response: {body[:500]}"
        raise requests.HTTPError(msg, response=r) from e
    return r.json()


def fetch_schema(endpoint: str = "/api/v1/conesearch") -> dict:
    """Retrieve the full data schema for a given Fink endpoint."""
    r = requests.post(
        f"{FINK_API}/api/v1/schema", json={"endpoint": endpoint, "output-format": "json"}, timeout=30
    )
    if r.status_code == 200:
        return r.json()
    print(f"schema HTTP {r.status_code}: {r.text[:200]}")
    return {}


def fetch_conesearch(
    ra: float,
    dec: float,
    radius: float,
    n: int = N_ALERTS_MAX,
    columns: str | None = COLUMNS_STR,
) -> pd.DataFrame:
    """
    Cone search via /api/v1/conesearch — returns one row per alert (diaSource).

    IMPORTANT: column names must use the 'r:' or 'f:' prefix.
    The 'i:' prefix is NOT supported here and causes HTTP 500 errors.

    Parameters
    ----------
    ra, dec  : field centre (degrees)
    radius   : search radius (arcsec)
    n        : max alerts to return
    columns  : comma-separated column string (None = all columns)

    Returns
    -------
    pd.DataFrame — one row per alert, or empty DataFrame on failure.
    """
    payload = {
        "ra": ra,
        "dec": dec,
        "radius": radius,
        "n": n,
        "output-format": "json",
    }

    # print("fetch_conesearch payload : ", payload)

    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/conesearch", payload)
        if not raw:
            return pd.DataFrame()
        return pd.DataFrame(raw)
    except Exception as e:
        print(f"fetch_conesearch ERROR (ra={ra:.3f}, dec={dec:.3f}): {e}")
        return pd.DataFrame()


def fetch_conesearch_sliced(
    ra: float,
    dec: float,
    radius: float,
    n: int,
    columns: str | None = COLUMNS_STR,
) -> pd.DataFrame:
    """Cone search fallback spatial adaptatif couvrant tout le cone."""
    dfs = []

    # 2) fallback spatial adaptatif
    tile_radius = min(600.0, max(250.0, radius / 3.5))  # arcsec
    step_arcsec = 0.95 * np.sqrt(2.0) * tile_radius
    nside = int(np.ceil((2.0 * radius) / step_arcsec)) + 1
    if nside % 2 == 0:
        nside += 1
    offs_arcsec = np.linspace(-radius, radius, nside)

    print(f"   fallback tiles: nside={nside}, tile_radius={tile_radius:.0f}, step~{step_arcsec:.0f}")

    tile_dfs = []
    for dx_as in offs_arcsec:
        for dy_as in offs_arcsec:
            # ignorer les centres trop loin du cone principal
            if np.hypot(dx_as, dy_as) > (radius + tile_radius):
                continue

            ra_i = ra + dx_as / 3600.0
            dec_i = dec + dy_as / 3600.0

            dfi = fetch_conesearch(
                ra_i,
                dec_i,
                tile_radius,
                N_ALERTS_MAX,
                columns,
            )

            if not dfi.empty:
                tile_dfs.append(dfi)
                nal = len(dfi)
                print(f"\t >>> tile with n_alerts = {nal}")
            time.sleep(0.15)

        if tile_dfs:
            dft = pd.concat(tile_dfs, ignore_index=True)
            if "r:diaSourceId" in dft.columns:
                dft = dft.drop_duplicates(subset="r:diaSourceId")
            dfs.append(dft)

    if not dfs:
        return pd.DataFrame()

    df_all = pd.concat(dfs, ignore_index=True)
    if "r:diaSourceId" in df_all.columns:
        df_all = df_all.drop_duplicates(subset="r:diaSourceId")
    return df_all


def fetch_sources(diaObjectId: int | str, columns: str | None = None) -> pd.DataFrame:
    """Fetch diaSources (direct detections) for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/sources", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_fp(diaObjectId: int | str, columns: str | None = None) -> pd.DataFrame:
    """Fetch forced photometry for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/fp", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_objects(diaObjectId: int | str, columns: str | None = None) -> pd.DataFrame:
    """Fetch aggregated object-level statistics via /api/v1/objects."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/objects", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_tags_stream(
    tag: str, n: int = 5000, startdate: str | None = None, columns: str | None = None
) -> pd.DataFrame:
    """
    Fetch recent alerts carrying a given Fink tag via /api/v1/tags.

    Useful tags:
      'cataloged'       → objects matched to an external catalogue
      'variablestar'    → Fink-classified variable stars (if tag exists)
      'roid'            → solar system objects

    Parameters
    ----------
    tag       : str  — Fink tag string (see /api/v1/tags for list)
    n         : int  — max number of alerts to return
    startdate : str  — ISO date string, e.g. '2025-09-01 00:00:00'
    columns   : str  — comma-separated column names (None = all)
    """
    payload = {"tag": tag, "n": n, "output-format": "json"}
    if startdate:
        payload["startdate"] = startdate
    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/tags", payload)
        return pd.DataFrame(raw) if raw else pd.DataFrame()
    except Exception as e:
        print(f"fetch_tags_stream({tag!r}): {e}")
        return pd.DataFrame()


def fetch_resolver_simbad(name: str, nmax: int = 5) -> pd.DataFrame:
    """
    Resolve an astronomical name via SIMBAD to find the corresponding LSST diaObjectId.

    Parameters
    ----------
    name : str  — SIMBAD name (e.g. 'delta Cep', 'RS Pup', 'eta Aql')
    nmax : int  — max number of matches

    Returns DataFrame with columns: diaObjectId, r:ra, r:dec, ...
    """
    payload = {"resolver": "simbad", "name_or_id": name, "nmax": nmax, "output-format": "json"}
    try:
        raw = _post_json(f"{FINK_API}/api/v1/resolver", payload)
        return pd.DataFrame(raw) if raw else pd.DataFrame()
    except Exception as e:
        print(f"fetch_resolver_simbad({name!r}): {e}")
        return pd.DataFrame()


def fetch_available_tags() -> list:
    """GET the list of available Fink tags."""
    try:
        r = requests.get(f"{FINK_API}/api/v1/tags", timeout=30)
        r.raise_for_status()
        data = r.json()
        return data if isinstance(data, list) else [data]
    except Exception as e:
        print(f"fetch_available_tags: {e}")
        return []


print("API wrappers defined.")

## 3. Inspect the API: schema and available tags

In [ ]:
# ── 3a. Check available Fink tags ─────────────────────────────────────────────
tags_list = fetch_available_tags()
print(f"Available Fink tags ({len(tags_list)}):")
for t in tags_list:
    print(f"  {t}")

In [ ]:
# ── 3b. Query the schema for conesearch — see EXACT column names available ────
schema = fetch_schema("/api/v1/conesearch")
if schema:
    print("Schema for /api/v1/conesearch:")
    if isinstance(schema, dict):
        for k, v in schema.items():
            print(f"  {k}: {str(v)[:120]}")
    elif isinstance(schema, list):
        for item in schema:
            print(f"  {item}")
    else:
        print(schema)
else:
    print("Schema not available.")

In [ ]:
# ── 3c. Probe a tiny conesearch on COSMOS to see ALL actual column names ──────
print("Probing COSMOS (r=60 arcsec, n=20) to read column names...")
df_probe = fetch_conesearch(ra=150.1191, dec=2.2058, radius=60.0, n=20)
if not df_probe.empty:
    print(f"\n{len(df_probe)} alerts, {len(df_probe.columns)} columns")
    print("\nALL column names:")
    for c in sorted(df_probe.columns):
        print(f"  {c}")

    print("\nCrossmatch-related columns:")
    xm_cols = [
        c
        for c in df_probe.columns
        if any(
            k in c.lower()
            for k in [
                "xm_",
                "cds",
                "simbad",
                "gcvs",
                "vsx",
                "clf",
                "block",
                "otype",
                "tag",
                "tns",
                "gaia",
                "mangrove",
                "legacy",
                "spicy",
                "3hsp",
                "4lac",
                "sso",
                "catalog",
            ]
        )
    ]
    for c in xm_cols:
        sample = df_probe[c].dropna().head(3).tolist()
        print(f"  {c:45s} → sample: {sample}")
else:
    print("No data returned.")

## 4. Fetch `cataloged` tag stream — all catalogue-matched objects

In [ ]:
# The 'cataloged' tag identifies alerts matched to any external catalogue.
# This is independent of cone-search and covers all fields observed by Rubin/LSST.
# We retrieve the maximum amount and then filter for pulsating variable types.

print("Fetching 'cataloged' tag stream (n=10000) ...")
df_cat = fetch_tags_stream(
    tag="cataloged",
    n=10000,
    startdate="2025-09-01 00:00:00",
)

if df_cat.empty:
    print("'cataloged' tag returned no data.")
    print("Check available tags above and replace with the correct tag name.")
else:
    print(f"\n{len(df_cat)} alerts retrieved from 'cataloged' tag stream")
    print(
        f"Unique diaObjectIds: {df_cat['r:diaObjectId'].nunique() if 'r:diaObjectId' in df_cat.columns else '?'}"
    )
    print(f"Columns ({len(df_cat.columns)}): {list(df_cat.columns)}")

In [ ]:
# Inspect crossmatch columns in the tag stream data
if not df_cat.empty:
    xm_cols = [
        c
        for c in df_cat.columns
        if any(k in c.lower() for k in ["xm_", "cds", "simbad", "gcvs", "vsx", "otype", "gaia", "catalog"])
    ]
    print("Crossmatch columns in tag stream:")
    for c in xm_cols:
        sample = df_cat[c].value_counts(dropna=False).head(10)
        print(f"\n  {c}:")
        print(sample.to_string())

## 5. Extended cone-search: all fields including galactic plane fields

In [ ]:
# Re-run cone-searches on ALL fields including galactic ones.
# Cepheids in the Milky Way live preferentially near the galactic plane.
# Carina (l~287, b~0) and Trifid-Lagoon (l~6, b~-1) should contain many.

# Check if we already have cached parquet files from previous run
all_alerts_list = []

for field_name, (ra, dec, radius, ftype) in ALL_FIELDS.items():
    cache_path = os.path.join(DIR_DATA, f"{field_name}_alerts.parquet")
    radius_deg = radius / 3600.0

    # Try to load from cache first
    if os.path.exists(cache_path):
        try:
            df_cone = pd.read_parquet(cache_path)
            print(
                f"[CACHE] {field_name:18s} ({ftype:14s}, r={radius_deg:.1f}°) "
                f"→ {len(df_cone):6d} alerts, "
                f"{df_cone['r:diaObjectId'].nunique():5d} objects"
            )
            all_alerts_list.append(df_cone)
            continue
        except Exception as e:
            print(f"[CACHE read error] {field_name}: {e} — re-fetching")

    # Fetch from API
    print(f"[API]   {field_name:18s} ({ftype:14s}, r={radius_deg:.1f}°) ...", end=" ", flush=True)
    try:
        t0 = time.time()
        df_cone = fetch_conesearch(ra, dec, radius=radius, n=N_ALERTS_MAX)
        elapsed = time.time() - t0

        if df_cone.empty:
            print(f"→ {field_name} :: NO DATA  ({elapsed:.1f}s), try with fetch_conesearch_sliced")
            df_cone = fetch_conesearch_sliced(ra, dec, radius=radius, n=N_ALERTS_MAX)
            elapsed = time.time() - t0
            ndata_fetch_conesearch_sliced = len(df_cone)
            print(
                f"→ {field_name} :: DATA with fetch_conesearch_sliced ({elapsed:.1f}s), ntot = {ndata_fetch_conesearch_sliced}"
            )
            if ndata_fetch_conesearch_sliced == 0:
                continue

        ndata = len(df_cone)
        print(f"{field_name} :: n_alerts = {ndata}")
        df_cone["field"] = field_name
        df_cone["field_type"] = ftype
        # Save to parquet for caching
        df_cone.to_parquet(cache_path, index=False)
        all_alerts_list.append(df_cone)
        print(
            f"=== {field_name} :: {len(df_cone):6d} alerts, {df_cone['r:diaObjectId'].nunique():5d} objects ==="
        )
    except Exception as e:
        print(f"ERROR: {e}")
    time.sleep(0.5)

if not all_alerts_list:
    raise RuntimeError("No data retrieved from any field.")

df_all = pd.concat(all_alerts_list, ignore_index=True)
print(f"\n{'=' * 60}")
print(f"TOTAL alerts   : {len(df_all):,}")
print(f"Unique objects : {df_all['r:diaObjectId'].nunique():,}")
print(f"Columns        : {len(df_all.columns)}")

## 6. Inspect all actual crossmatch column values in the data

In [ ]:
# Deduplicate by diaObjectId BEFORE inspecting types
df_obj = (
    df_all.sort_values("r:midpointMjdTai")
    .drop_duplicates(subset="r:diaObjectId", keep="last")
    .reset_index(drop=True)
)
print(f"Unique objects before nDiaSources cut: {len(df_obj):,}")

# ── Cut on nDiaSources >= NSRC_MIN ──────────────────────────────────────────────
# This dramatically reduces the number of objects for which we download
# full light curves, keeping only well-sampled sources.
if "r:nDiaSources" in df_obj.columns:
    df_obj["r:nDiaSources"] = pd.to_numeric(df_obj["r:nDiaSources"], errors="coerce")
    n_before = len(df_obj)
    df_obj = df_obj[df_obj["r:nDiaSources"] >= NSRC_MIN].reset_index(drop=True)
    print(f"After nDiaSources >= {NSRC_MIN}: {len(df_obj):,} objects (removed {n_before - len(df_obj):,})")
else:
    print("WARNING: 'r:nDiaSources' column not found — no nDiaSources cut applied.")

print(f"\nFields covered:")
print(df_obj["field"].value_counts().to_string())

In [ ]:
# ── Print value distributions for ALL crossmatch columns ─────────────────────
xm_candidates = [
    c
    for c in df_obj.columns
    if any(
        k in c.lower()
        for k in [
            "xm_",
            "cds",
            "simbad",
            "gcvs",
            "vsx",
            "otype",
            "tns",
            "gaia",
            "mangrove",
            "legacy",
            "spicy",
            "3hsp",
            "4lac",
            "catalog",
            "block",
            "tag",
            "clf",
            "sso",
        ]
    )
]

print(f"Found {len(xm_candidates)} crossmatch/classification columns:")
for c in sorted(xm_candidates):
    vc = df_obj[c].value_counts(dropna=False)
    n_nonnull = (
        df_obj[c].notna() & (df_obj[c].astype(str) != "nan") & (df_obj[c].astype(str) != "None")
    ).sum()
    print(f"\n  ── {c}  ({n_nonnull} non-null) ──")
    print(vc.head(20).to_string())

## 7. Cepheid & pulsating variable selection — multi-column approach

In [ ]:
def is_null(val) -> bool:
    """Return True if the value is a null/empty/sentinel string."""
    return val is None or str(val).strip() in ("", "None", "nan", "Fail", "null", "N/A")


def classify_pulsator(row) -> str:
    """
    Classify an alert row as a pulsating variable type.

    Returns one of:
      'cepheid_classical'  — delta Cep type (DCEP, CEP, CW...)
      'cepheid_type2'      — W Vir, CWA, CWB, BLBOO
      'rr_lyrae'           — RR Lyrae (same instability strip)
      'delta_scuti'        — delta Scuti, SX Phe (short period)
      'lpv_mira'           — Mira / LPV / semi-regular
      'other_pulsator'     — other periodic pulsator
      'not_pulsator'       — not a pulsating variable
    """
    # Read all relevant crossmatch columns
    simbad = str(row.get("f:xm_simbad_otype", "") or "").strip()
    gcvs = str(row.get("f:xm_gcvs_type", "") or "").strip().upper()
    vsx = str(row.get("f:xm_vsx_Type", "") or "").strip().upper()
    # Try also any cdsxmatch variant
    cds = ""
    for col in ("cdsxmatch", "f:cdsxmatch", "r:cdsxmatch"):
        if not is_null(row.get(col, None)):
            cds = str(row[col]).strip()
            break

    # ── Classical Cepheids ─────────────────────────────────────────────────────
    if (
        simbad in {"Cep", "deltaCep", "bCep"}
        or vsx in {"CEP", "CEP(B)", "DCEP", "DCEPS", "DCEP(B)", "BCEP"}
        or gcvs in {"CEP", "DCEP", "DCEPS", "CEP(B)"}
        or cds in {"Cepheid", "Classical Cepheid", "deltaCep", "Cep", "bCep"}
    ):
        return "cepheid_classical"

    # ── Type II Cepheids ───────────────────────────────────────────────────────
    if (
        simbad in {"WVir"}
        or vsx in {"CW", "CWA", "CWB", "BLBOO", "ACEP", "ACEP(B)", "WVIR"}
        or gcvs in {"CW", "CWA", "CWB", "BLBOO", "ACEP"}
        or cds in {"Type II Cepheid", "WVir"}
    ):
        return "cepheid_type2"

    # ── RR Lyrae ───────────────────────────────────────────────────────────────
    if (
        simbad in {"RRLyr"}
        or vsx in {"RRAB", "RRC", "RRD", "RR"}
        or gcvs in {"RRAB", "RRC", "RRD", "RR"}
        or cds in {"RRLyr"}
    ):
        return "rr_lyrae"

    # ── Delta Scuti / SX Phe ──────────────────────────────────────────────────
    if (
        simbad in {"dS*", "SX*"}
        or vsx in {"DSCT", "DSCT(B)", "SXPHE", "SXPHE(B)"}
        or gcvs in {"DSCT", "SXPHE"}
    ):
        return "delta_scuti"

    # ── Mira / LPV / semi-regular ─────────────────────────────────────────────
    if (
        simbad in {"Mira", "LPV*", "SRS"}
        or vsx in {"M", "SR", "SRA", "SRB", "SRC", "SRD"}
        or gcvs in {"M", "SR", "SRA", "SRB", "SRC", "SRD"}
    ):
        return "lpv_mira"

    # ── RV Tauri ──────────────────────────────────────────────────────────────
    if simbad in {"RV*"} or vsx in {"RVA", "RVB", "RV"} or gcvs in {"RVA", "RVB"}:
        return "rv_tauri"

    # ── Generic pulsating variable ────────────────────────────────────────────
    if simbad in {"V*"} or not is_null(vsx) or not is_null(gcvs):
        return "other_pulsator"

    return "not_pulsator"


# Apply classifier
df_obj["pulsator_class"] = df_obj.apply(classify_pulsator, axis=1)
print("Pulsator classification (all objects):")
print(df_obj["pulsator_class"].value_counts().to_string())

In [ ]:
# ── Strict Cepheid selection ───────────────────────────────────────────────────
df_cepheids = (
    df_obj[df_obj["pulsator_class"].isin(["cepheid_classical", "cepheid_type2"])]
    .copy()
    .reset_index(drop=True)
)

# ── Broad instability-strip selection (includes RR Lyr, delta Sct, etc.) ──────
df_pulsators = df_obj[df_obj["pulsator_class"] != "not_pulsator"].copy().reset_index(drop=True)

print(f"Strict Cepheids    : {len(df_cepheids)}")
print(f"All pulsators      : {len(df_pulsators)}")
print()

if not df_pulsators.empty:
    print("Pulsators per field:")
    print(
        df_pulsators.groupby(["field", "pulsator_class"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .to_string(index=False)
    )

In [ ]:
# ── If strict Cepheids found: show catalogue ──────────────────────────────────
if not df_cepheids.empty:
    print(f"\nCepheid catalogue ({len(df_cepheids)} objects):")
    show_cols = ["r:diaObjectId", "r:ra", "r:dec", "r:nDiaSources", "field", "pulsator_class"]
    for c in ("f:xm_simbad_otype", "f:xm_gcvs_type", "f:xm_vsx_Type", "cdsxmatch", "f:cdsxmatch"):
        if c in df_cepheids.columns:
            show_cols.append(c)
    display(df_cepheids[show_cols])

    df_cepheids.to_csv(os.path.join(DIR_DATA, "cepheids_strict.csv"), index=False)
    print("Saved: cepheids_strict.csv")
else:
    print("No strict Cepheids found.")
    if not df_pulsators.empty:
        print("Using all pulsators for subsequent analysis.")
        df_cepheids = df_pulsators.copy()  # Fallback

if not df_pulsators.empty:
    df_pulsators.to_csv(os.path.join(DIR_DATA, "pulsators_all.csv"), index=False)
    print("Saved: pulsators_all.csv")

## 8. Resolver: search known GCVS Cepheids within the LSST footprint

- Known bright Cepheids in or near the southern LSST sky
- These are GCVS/SIMBAD names — the resolver tries to find their diaObjectId
- if they were detected by Rubin/LSST

In [ ]:
# Known bright Cepheids in or near the southern LSST sky
# These are GCVS/SIMBAD names — the resolver tries to find their diaObjectId
# if they were detected by Rubin/LSST
KNOWN_CEPHEIDS_SOUTH = [
    # Very bright / famous
    "l Car",  # P=35.5 d, V=3.7, in Carina (dec=-62)
    "RS Pup",  # P=41.5 d, V=7.0, dec=-34
    "beta Dor",  # P=9.84 d, V=3.7, dec=-62
    "kappa Pav",  # P=9.09 d, Type II Cepheid, dec=-67
    "T Mon",  # P=27.0 d, dec=+7 (barely south)
    # In/near the Carina or galactic field
    "U Car",  # P=38.9 d, dec=-59
    "GH Car",  # dec=-59
    "XX Car",  # dec=-60
    "XY Car",  # dec=-59
    "V Cen",  # P=5.49 d, dec=-56
    # RR Lyrae in the south (same instability strip)
    "RR Lyr",  # prototype — too north (dec=+42), expected not found
]

resolver_results = []
print("Querying Fink resolver for known southern Cepheids...")
for name in KNOWN_CEPHEIDS_SOUTH:
    df_res = fetch_resolver_simbad(name, nmax=3)
    if not df_res.empty:
        df_res["query_name"] = name
        resolver_results.append(df_res)
        print(f"  {name:15s} → {len(df_res)} match(es): {df_res.to_dict('records')[:1]}")
    else:
        print(f"  {name:15s} → not found in Fink/LSST")
    time.sleep(0.2)

if resolver_results:
    df_resolved = pd.concat(resolver_results, ignore_index=True)
    print(f"\nResolved {len(df_resolved)} matches for known Cepheids.")
    display(df_resolved)
    df_resolved.to_csv(os.path.join(DIR_DATA, "known_cepheids_resolved.csv"), index=False)
else:
    df_resolved = pd.DataFrame()
    print("No known Cepheids found in the Fink/LSST database.")
    print("This is expected if the galactic fields were not yet observed,")
    print("or if these specific stars fell outside the detector footprint.")

## 9. Explore the distribution of all variable types actually in the data

In [ ]:
# This cell gives us the ground truth: what types of variable stars are
# actually present in the crossmatched Fink/LSST data right now.
# Plot the top types from each crossmatch source.

fig, axes = plt.subplots(1, 3, figsize=(16, 5))


def plot_top_types(ax, col_name, title, top_n=25):
    if col_name not in df_obj.columns:
        ax.text(
            0.5,
            0.5,
            f"{col_name}\nnot available",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=9,
        )
        ax.set_title(title)
        return
    vc = (
        df_obj[col_name]
        .dropna()
        .astype(str)
        .replace({"nan": np.nan, "None": np.nan, "Fail": np.nan})
        .dropna()
        .value_counts()
        .head(top_n)
    )
    if vc.empty:
        ax.text(0.5, 0.5, "All null", ha="center", va="center", transform=ax.transAxes)
    else:
        vc.sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_xlabel("Count")
    ax.set_title(title)


plot_top_types(axes[0], "f:xm_simbad_otype", "SIMBAD otype\n(f:xm_simbad_otype)")
plot_top_types(axes[1], "f:xm_gcvs_type", "GCVS type\n(f:xm_gcvs_type)")
plot_top_types(axes[2], "f:xm_vsx_Type", "VSX type\n(f:xm_vsx_Type)")

plt.suptitle(
    "Variable star types in Fink/LSST crossmatch data (all DDF + galactic fields)", y=1.01, fontsize=11
)
plt.tight_layout()
savefig("variable_types_distribution")
plt.show()

In [ ]:
# Sky map: all pulsators found
if not df_pulsators.empty:
    fig, ax = plt.subplots(figsize=(12, 5))

    class_colors = {
        "cepheid_classical": "red",
        "cepheid_type2": "darkorange",
        "rr_lyrae": "dodgerblue",
        "delta_scuti": "green",
        "lpv_mira": "purple",
        "rv_tauri": "brown",
        "other_pulsator": "grey",
    }
    for cls, color in class_colors.items():
        sub = df_pulsators[df_pulsators["pulsator_class"] == cls]
        if sub.empty:
            continue
        ax.scatter(
            sub["r:ra"].astype(float),
            sub["r:dec"].astype(float),
            s=20,
            color=color,
            alpha=0.8,
            label=f"{cls} ({len(sub)})",
            zorder=3,
        )

    for fname, (ra, dec, _, ftype) in ALL_FIELDS.items():
        ax.scatter(ra, dec, marker="+", s=200, color="black", lw=1.5, zorder=2)
        ax.text(ra + 0.5, dec + 0.3, fname, fontsize=6, color="dimgrey")

    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_title(f"Pulsating variables found in all LSST fields (N={len(df_pulsators)})")
    ax.legend(fontsize=7, ncol=2, loc="best")
    plt.tight_layout()
    savefig("pulsators_sky_map")
    plt.show()
else:
    print("No pulsators found to plot.")

## 10. Download light curves for selected Cepheids / pulsators

In [ ]:
AB_FLUX_ZERO_NJY = 3631e9


def flux_to_mag(flux_nJy, flux_err_nJy=None):
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO_NJY), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def filter_lc(
    df_lc,
    mjd_col="r:midpointMjdTai",
    flux_col="r:scienceFlux",
    ferr_col="r:scienceFluxErr",
    band_col="r:band",
    snr_min=SNR_MIN,
):
    """
    Apply SNR cut and add mag/mag_err columns to a sources DataFrame.

    parameters:
    ===========
        df_lc : the light curve either src or fp
        mjd_col : column name with the relevant mjd name
        flux_col : column name with the relevant flux (default the scienceFlux for variable stars)
        ferr_col : column name with the relevant flux err
        band_col : band
        snr_min : cut on SNR

    """

    df = df_lc.copy()
    for col in (mjd_col, flux_col, ferr_col, band_col):
        if col not in df.columns:
            return pd.DataFrame()

    df[flux_col] = pd.to_numeric(df[flux_col], errors="coerce")
    df[ferr_col] = pd.to_numeric(df[ferr_col], errors="coerce")
    df[mjd_col] = pd.to_numeric(df[mjd_col], errors="coerce")

    snr = df[flux_col].abs() / df[ferr_col].replace(0, np.nan)

    df = df[snr >= snr_min].sort_values(mjd_col).reset_index(drop=True)
    df = df.dropna(subset=[flux_col, ferr_col, mjd_col]).reset_index(drop=True)

    mag, mag_err = flux_to_mag(df[flux_col].values, df[ferr_col].values)
    df["mag"] = mag
    df["mag_err"] = mag_err

    df = df[np.isfinite(df["mag"].values)].copy()
    if df.empty:
        return df
    # return df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)
    return df.sort_values("r:midpointMjdTai").reset_index(drop=True)


def lomb_scargle_period(mjd, mag, mag_err, pmin=PERIOD_MIN_DAYS, pmax=PERIOD_MAX_DAYS, samples=LS_SAMPLES):
    """
    calculate the LombScargle spectrum from astropy on src magnitudes only

    Parameters:
    ----------
        mjd: array of Modified julina days
        mag: magnitudes
        mag_err : magnitude errors

    Returns:
    -------
         best period
         1D- array of periods:
         1D-array of power
         fap : False alaem on periodic probability
    """

    # clean the nan
    mask = np.isfinite(mjd) & np.isfinite(mag) & np.isfinite(mag_err) & (mag_err > 0)

    # extract the dates and y values for the astropy LombScargle
    t, y, dy = mjd[mask], mag[mask], mag_err[mask]

    if len(t) < 5:
        return np.nan, np.array([]), np.array([]), np.nan

    # call LombScargle analysis
    ls = LombScargle(t, y, dy)

    # extract the frequency and power of the spectrum
    frequency, power = ls.autopower(
        minimum_frequency=1.0 / pmax,
        maximum_frequency=1.0 / pmin,
        samples_per_peak=samples,
    )

    # calculate the best frequency or period
    best_freq = frequency[np.argmax(power)]
    best_period = 1.0 / best_freq

    # calculate tje false alarm probability
    try:
        fap = ls.false_alarm_probability(power.max(), method="baluev")
    except Exception:
        fap = np.nan
    return best_period, 1.0 / frequency, power, fap


def lomb_scargle_period_multiband(
    df_src: "pd.DataFrame",
    pmin: float = PERIOD_MIN_DAYS,
    pmax: float = PERIOD_MAX_DAYS,
    samples_per_peak: int = LS_SAMPLES,
):
    """
    Multiband Lomb-Scargle period search on a filtered sources DataFrame.

    Uses astropy.timeseries.LombScargleMultiband when more than one band is
    present, which fits a shared frequency with per-band amplitude/phase offsets.
    Falls back to single-band LombScargle when only one band is available.

    Parameters
    ----------
    df_src           : filtered sources DataFrame (output of filter_lc),
                       must contain columns r:midpointMjdTai, mag, mag_err,
                       r:band.
    pmin, pmax       : period search range in days.
    samples_per_peak : frequency grid oversampling factor.

    Returns
    -------
    best_period : float (days) or np.nan
    freq        : 1-D array of frequencies (1/day) or None
    power       : 1-D array of LS power or None
    fap         : false-alarm probability (bootstrap) or np.nan
    n_bands     : int — number of bands actually used
    """
    if df_src.empty or len(df_src) < 5:
        return np.nan, None, None, np.nan, 0

    # extract usefull finite datapoints
    t = df_src["r:midpointMjdTai"].values
    mag = df_src["mag"].values
    mag_err = df_src["mag_err"].values
    bands = df_src["r:band"]

    # Filter bad points before counting bands and running the period search.
    mask = np.isfinite(t) & np.isfinite(mag) & np.isfinite(mag_err) & (mag_err > 0)
    t_arr, y_arr, dy_arr, b_arr = t[mask], mag[mask], mag_err[mask], bands[mask]
    if len(t_arr) < 5:
        return np.nan, None, None, np.nan, 0

    fmin = 1.0 / pmax
    fmax = 1.0 / pmin

    # Keep only bands with enough points to constrain a sinusoid.
    bands_ok = [b for b in BANDS if np.sum(b_arr == b) >= 3]
    n_bands = len(bands_ok)
    use_band = np.isin(b_arr, bands_ok)

    # Multiband path: one common frequency, with per-band offsets/amplitudes.
    if n_bands >= 2 and np.sum(use_band) >= 5:
        try:
            lsm = LombScargleMultiband(
                t_arr[use_band],
                y_arr[use_band],
                b_arr[use_band],
                dy_arr[use_band],
                nterms_base=1,
                nterms_band=1,
            )
            freq, power = lsm.autopower(
                minimum_frequency=fmin,
                maximum_frequency=fmax,
                samples_per_peak=samples_per_peak,
            )
            best_freq = freq[np.argmax(power)]
            best_period = 1.0 / best_freq if best_freq > 0 else np.nan

            # Astropy does not provide a direct FAP for LombScargleMultiband.
            # The retained FAP is selected later from independent per-band scans.
            fap = np.nan

            return best_period, freq, power, fap, n_bands
        except Exception as exc:
            # Multiband failed; fall through to the single-band fallback.
            print(f"    [LS multiband fallback] {exc}")

    # Single-band fallback: prefer r-band; use all bands only if r is too sparse.
    if "r:band" in df_src.columns:
        df_r = df_src[df_src["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_src
    else:
        df_r = df_src

    if len(df_r) < 5:
        return np.nan, None, None, np.nan, 1

    best_period, period, power, fap = lomb_scargle_period(
        df_r["r:midpointMjdTai"].values,
        df_r["mag"].values,
        df_r["mag_err"].values,
        pmin=pmin,
        pmax=pmax,
        samples=samples_per_peak,
    )
    freq = 1.0 / period if len(period) else np.array([])

    return best_period, freq, power, fap, 1


def lomb_scargle_period_optimizeinallbands(
    df_src: "pd.DataFrame",
    pmin: float = PERIOD_MIN_DAYS,
    pmax: float = PERIOD_MAX_DAYS,
    samples_per_peak: int = LS_SAMPLES,
):
    """
    Run independent single-band Lomb-Scargle searches for each usable band.

    Parameters
    ----------
    df_src           : filtered sources DataFrame (output of filter_lc),
                       must contain columns r:midpointMjdTai, mag, mag_err,
                       r:band.
    pmin, pmax       : period search range in days.
    samples_per_peak : frequency grid oversampling factor.

    Returns
    -------
    pd.DataFrame with one row per band and columns band_ok, band_period,
    band_fap, and band_npoints.
    """
    if df_src.empty or len(df_src) < 5 or "r:band" not in df_src.columns:
        return pd.DataFrame(columns=["band_ok", "band_period", "band_fap", "band_npoints"])

    band_select = []
    band_period = []
    band_fap = []
    band_npoints = []

    # Run one ordinary Lomb-Scargle search per band with enough filtered points.
    for band in BANDS:
        df_b = df_src[df_src["r:band"] == band]
        if len(df_b) < 5:
            continue

        best_period_b, _, _, fap_b = lomb_scargle_period(
            df_b["r:midpointMjdTai"].values,
            df_b["mag"].values,
            df_b["mag_err"].values,
            pmin=pmin,
            pmax=pmax,
            samples=samples_per_peak,
        )

        band_select.append(band)
        band_period.append(best_period_b)
        band_fap.append(fap_b)
        band_npoints.append(len(df_b))

    results = {
        "band_ok": band_select,
        "band_period": band_period,
        "band_fap": band_fap,
        "band_npoints": band_npoints,
    }

    return pd.DataFrame(results)


def phase_fold(mjd, period, t0=None):
    if t0 is None:
        t0 = np.nanmin(mjd)
    return ((mjd - t0) / period) % 1.0


print("Utility functions defined.")

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """
    Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'.

    Uses astropy.time.Time for the conversion.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 7) -> None:
    """
    Add a secondary x-axis on **top** of *ax* showing calendar dates (YYYY-MM-DD),
    inclined 40 degrees to the left for readability.
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return

    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return

    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=12, labelpad=6)


print("mjd_to_datestr() and add_date_axis_on_top() defined.")

In [ ]:
# Load or download sources + forced photometry for all selected objects.
# Use df_cepheids (strict Cepheids if found, else all pulsators).
target_df = df_cepheids if not df_cepheids.empty else df_pulsators

LC_DIR = os.path.join(DIR_DATA, "lightcurves")
os.makedirs(LC_DIR, exist_ok=True)


def load_light_curve_from_disk(oid):
    """Load one cached light curve from disk, if parquet files are available."""
    obj_dir = os.path.join(LC_DIR, str(oid))
    src_path = os.path.join(obj_dir, "sources.parquet")
    fp_path = os.path.join(obj_dir, "fp.parquet")

    if not os.path.exists(src_path) and not os.path.exists(fp_path):
        return None

    df_src = pd.read_parquet(src_path) if os.path.exists(src_path) else pd.DataFrame()
    df_fp = pd.read_parquet(fp_path) if os.path.exists(fp_path) else pd.DataFrame()
    return df_src, df_fp


def save_light_curve_to_disk(oid, df_src, df_fp):
    """Persist one light curve immediately so later runs can skip the API call."""
    obj_dir = os.path.join(LC_DIR, str(oid))
    os.makedirs(obj_dir, exist_ok=True)
    if not df_src.empty:
        df_src.to_parquet(os.path.join(obj_dir, "sources.parquet"), index=False)
    if not df_fp.empty:
        df_fp.to_parquet(os.path.join(obj_dir, "fp.parquet"), index=False)


if target_df.empty:
    print("No pulsators found at all — cannot load or download light curves.")
    print("See Section 11 for diagnostic plots of the full sample.")
else:
    lc_dict = {}
    n_loaded = 0
    n_downloaded = 0
    ids_to_fetch = target_df["r:diaObjectId"].tolist()
    print(f"Loading/downloading light curves for {len(ids_to_fetch)} objects...")
    print(f"Cache directory: {os.path.abspath(LC_DIR)}")

    for i, oid in enumerate(ids_to_fetch):
        meta = target_df[target_df["r:diaObjectId"] == oid].iloc[0]
        source_label = "API"
        try:
            cached = load_light_curve_from_disk(oid)
            if cached is None:
                df_src = fetch_sources(oid)
                df_fp = fetch_fp(oid)
                save_light_curve_to_disk(oid, df_src, df_fp)
                n_downloaded += 1
            else:
                df_src, df_fp = cached
                source_label = "DISK"
                n_loaded += 1

            # Keep raw light curves in memory; filtering is done later.
            lc_dict[oid] = {"src": df_src, "fp": df_fp, "meta": meta}
            print(
                f"  [{i + 1:3d}/{len(ids_to_fetch)}] {source_label:4s} {oid}  "
                f"{len(df_src) if not df_src.empty else 0:4d} src  "
                f"{len(df_fp) if not df_fp.empty else 0:5d} fp  "
                f"({meta.get('pulsator_class', '?')} | {meta.get('field', '?')})"
            )
        except Exception as e:
            print(f"  [{i + 1:3d}] {oid}  ERROR: {e}")
            lc_dict[oid] = {"src": pd.DataFrame(), "fp": pd.DataFrame(), "meta": meta}
        if source_label == "API":
            time.sleep(0.3)

    print(f"\nLoaded {n_loaded} light curves from disk.")
    print(f"Downloaded {n_downloaded} light curves from the API.")
    print(f"Available in memory: {len(lc_dict)} light curves.")

## 11. Raw light curves — overview plot

In [ ]:
NCURVESMAX = 50

if not lc_dict:
    print("No light curves available.")
else:
    NC_PLOT = min(NCURVESMAX, len(lc_dict))
    ncols = 2
    nrows = int(np.ceil(NC_PLOT / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    # loop on a bunch of curves not preselected in advance, use only the science flux from sources, not the fp.
    for idx, (oid, data) in enumerate(list(lc_dict.items())[:NC_PLOT]):
        ax = axes[idx]
        meta = data["meta"]
        df_filt = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()

        if df_filt.empty:
            ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        else:
            mag = df_filt["mag"].values
            mag = mag[np.isfinite(mag)]
            mag_min = np.percentile(mag, 5) - 0.3
            mag_max = np.percentile(mag, 95) + 0.3

            for band in BANDS:
                dfb = df_filt[df_filt["r:band"] == band]
                if dfb.empty:
                    continue
                ax.errorbar(
                    dfb["r:midpointMjdTai"],
                    dfb["mag"],
                    dfb["mag_err"],
                    fmt="o",
                    ms=2,
                    lw=0.5,
                    color=BAND_COLORS.get(band, "grey"),
                    label=band,
                    alpha=0.8,
                )
            # ax.invert_yaxis()
            ax.set_ylim(mag_max, mag_min)
            add_date_axis_on_top(ax, df_filt["r:midpointMjdTai"])

        pclass = meta.get("pulsator_class", "?")
        stype = meta.get("f:xm_simbad_otype", "?")
        vsx = meta.get("f:xm_vsx_Type", "?")
        ax.set_title(f"{oid}\n{pclass} | SIMBAD:{stype} VSX:{vsx}", fontsize=7)
        ax.set_xlabel("MJD", fontsize=7)
        ax.set_ylabel("AB mag", fontsize=7)
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[NC_PLOT:]:
        ax.set_visible(False)

    plt.suptitle("Raw light curves — selected Cepheids/pulsators", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_raw_lc")
    plt.show()

## 12. Period search + phase-folded light curves

$$\Delta t _{phase}= t-P \times E\left(\frac{t}{P}\right)$$

In [ ]:
period_results = []

for oid, data in lc_dict.items():
    if data["src"].empty:
        continue

    df_filt = filter_lc(data["src"])
    if len(df_filt) < 100:
        continue
    meta = data["meta"]

    # print(f"--------------- {oid} --------------------------------")

    # Prefer the period found by LombScargleMultiband.
    best_period, _, _, _, n_bands = lomb_scargle_period_multiband(df_filt)

    dflombsc_results = lomb_scargle_period_optimizeinallbands(df_filt)
    # display(dflombsc_results)

    valid_fap = pd.to_numeric(dflombsc_results.get("band_fap", pd.Series(dtype=float)), errors="coerce")
    valid_fap = valid_fap[np.isfinite(valid_fap) & (valid_fap > 0)]
    fap = valid_fap.min() if len(valid_fap) else np.nan

    period_results.append(
        {
            "diaObjectId": oid,
            "field": meta.get("field", ""),
            "pulsator_class": meta.get("pulsator_class", ""),
            "simbad_otype": meta.get("f:xm_simbad_otype", ""),
            "vsx_type": meta.get("f:xm_vsx_Type", ""),
            "gcvs_type": meta.get("f:xm_gcvs_type", ""),
            "best_period_d": best_period,
            "ls_fap": fap,
            "n_pts": len(df_filt),
            "n_bands": n_bands,
        }
    )

    print(f" {oid} :: P={best_period:.3f}d  FAP={fap:.1e}  ({meta.get('pulsator_class', '?')})")
    # print("----------------------------------------------")

df_periods = pd.DataFrame(period_results)
if not df_periods.empty:
    df_periods.to_csv(os.path.join(DIR_DATA, "pulsators_periods.csv"), index=False)
    print(f"\n{len(df_periods)} periods computed. Saved pulsators_periods.csv")
    display(df_periods.sort_values("best_period_d"))

### Phase diagram

In [ ]:
# Phase-folded plots for the most significant periods
if not df_periods.empty:
    df_plot = df_periods.dropna(subset=["best_period_d"]).sort_values("ls_fap").head(120)
    ncols = 4
    nrows = int(np.ceil(len(df_plot) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (_, row) in enumerate(df_plot.iterrows()):
        oid = row["diaObjectId"]
        period = row["best_period_d"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if df_filt.empty:
            continue

        ax = axes[idx]
        t0 = df_filt["r:midpointMjdTai"].min()
        for band in BANDS:
            dfb = df_filt[df_filt["r:band"] == band]
            if dfb.empty:
                continue
            phi = phase_fold(dfb["r:midpointMjdTai"].values, period, t0)
            ax.errorbar(
                np.concatenate([phi, phi + 1]),
                np.concatenate([dfb["mag"].values] * 2),
                yerr=np.concatenate([dfb["mag_err"].values] * 2),
                fmt="o",
                ms=3,
                lw=0.5,
                color=BAND_COLORS.get(band, "grey"),
                label=band,
                alpha=0.85,
            )
        ax.set_xlim(0, 2)
        ax.invert_yaxis()
        ax.set_xlabel("Phase")
        ax.set_ylabel("AB mag")
        ax.set_title(
            f"{oid}  P={period:.3f}d  FAP={row['ls_fap']:.1e}\n"
            f"{row['pulsator_class']} | {row.get('vsx_type', '?')}",
            fontsize=7,
        )
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[len(df_plot) :]:
        ax.set_visible(False)

    plt.suptitle("Phase-folded light curves (sorted by LS FAP)", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_phased_lc")
    plt.show()

## 12b. RR Lyrae and Delta Scuti phase plots

In [ ]:
# Plot RR Lyrae and Delta Scuti objects with one subplot per object.
# Each subplot shows phase-folded flux on the left axis and phase-folded magnitude on the right axis.
TARGET_PULSATOR_CLASSES = {"rr_lyrae", "delta_scuti"}
FLUX_COL = "r:scienceFlux"
FLUX_ERR_COL = "r:scienceFluxErr"

if df_periods.empty:
    print("No period results available for RR Lyrae / Delta Scuti plots.")
else:
    df_rr_dsct = df_periods[
        df_periods["pulsator_class"].isin(TARGET_PULSATOR_CLASSES) & df_periods["best_period_d"].notna()
    ].copy()

    if df_rr_dsct.empty:
        print("No RR Lyrae or Delta Scuti objects with valid periods.")
    else:
        ncols = 4
        nrows = int(np.ceil(len(df_rr_dsct) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 4.2 * nrows), squeeze=False)
        axes = axes.flatten()

        for ax, (_, row) in zip(axes, df_rr_dsct.iterrows(), strict=False):
            oid = row["diaObjectId"]
            period = row["best_period_d"]
            data = lc_dict.get(oid, {})
            if not data or data["src"].empty:
                ax.text(0.5, 0.5, "No light curve", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue

            df_filt = filter_lc(data["src"])
            if df_filt.empty or FLUX_COL not in df_filt.columns:
                ax.text(0.5, 0.5, "No valid filtered data", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue

            t0 = df_filt["r:midpointMjdTai"].min()
            ax_mag = ax.twinx()

            for band in BANDS:
                dfb = df_filt[df_filt["r:band"] == band]
                if dfb.empty:
                    continue

                phase = phase_fold(dfb["r:midpointMjdTai"].values, period, t0)
                phase2 = np.concatenate([phase, phase + 1.0])
                color = BAND_COLORS.get(band, "grey")

                flux = pd.to_numeric(dfb[FLUX_COL], errors="coerce").values
                flux2 = np.concatenate([flux, flux])
                if FLUX_ERR_COL in dfb.columns:
                    flux_err = pd.to_numeric(dfb[FLUX_ERR_COL], errors="coerce").values
                    flux_err2 = np.concatenate([flux_err, flux_err])
                else:
                    flux_err2 = None

                ax.errorbar(
                    phase2,
                    flux2,
                    yerr=flux_err2,
                    fmt="o",
                    ms=3,
                    lw=0.5,
                    alpha=0.45,
                    color=color,
                    label=f"{band} flux",
                )

                ax_mag.scatter(
                    phase2,
                    np.concatenate([dfb["mag"].values, dfb["mag"].values]),
                    marker="x",
                    s=18,
                    alpha=0.8,
                    color=color,
                    label=f"{band} mag",
                )

            ax.set_xlim(0, 2)
            ax.grid(True, alpha=0.35)
            ax.set_xlabel("Phase")
            ax.set_ylabel("Science flux (nJy)")
            ax_mag.set_ylabel("AB mag")
            ax_mag.invert_yaxis()
            ax.set_title(
                f"{oid} | {row['pulsator_class']} | P={period:.4f} d | FAP={row['ls_fap']:.1e}",
                fontsize=8,
            )
            ax.legend(fontsize=6, ncol=3, loc="upper left")

        for ax in axes[len(df_rr_dsct) :]:
            ax.set_visible(False)

        plt.suptitle("RR Lyrae and Delta Scuti: phase-folded flux and light curves", y=1.01, fontsize=11)
        plt.tight_layout()
        savefig("rr_lyrae_delta_scuti_phase_flux_lightcurves")
        plt.show()

## 13. Diagnostic: why are Cepheids rare in DDF data?

This cell produces a diagnostic summary to understand the situation.

In [ ]:
print("=" * 65)
print("DIAGNOSTIC SUMMARY")
print("=" * 65)

print(f"""
1. TOTAL OBJECTS SURVEYED    : {len(df_obj):,}
   - In DDFs (extragalactic) : {len(df_obj[df_obj["field_type"] == "extragalactic"]):,}
   - In galactic fields      : {len(df_obj[df_obj["field_type"] == "galactic"]):,}
   - In SV test fields       : {len(df_obj[df_obj["field_type"] == "sv"]):,}

2. CROSSMATCH COVERAGE:
""")
for col in ("f:xm_simbad_otype", "f:xm_gcvs_type", "f:xm_vsx_Type"):
    if col in df_obj.columns:
        n_match = (
            df_obj[col].notna()
            & (df_obj[col].astype(str) != "nan")
            & (df_obj[col].astype(str) != "None")
            & (df_obj[col].astype(str) != "Fail")
        ).sum()
        print(f"   {col:35s}: {n_match:5d} matched ({100 * n_match / len(df_obj):.1f}%)")
    else:
        print(f"   {col:35s}: NOT PRESENT in API response")

print(f"""
3. PULSATING VARIABLES FOUND : {len(df_pulsators):,}
   - Strict Cepheids         : {len(df_obj[df_obj["pulsator_class"].isin(["cepheid_classical", "cepheid_type2"])]):,}
   - RR Lyrae                : {len(df_obj[df_obj["pulsator_class"] == "rr_lyrae"]):,}
   - Delta Scuti             : {len(df_obj[df_obj["pulsator_class"] == "delta_scuti"]):,}
   - LPV/Mira                : {len(df_obj[df_obj["pulsator_class"] == "lpv_mira"]):,}
   - Other pulsators         : {len(df_obj[df_obj["pulsator_class"] == "other_pulsator"]):,}

4. KNOWN CEPHEIDS VIA RESOLVER:
   Resolved matches          : {len(df_resolved) if not df_resolved.empty else 0}

5. INTERPRETATION:
   - Classical Cepheids (Pop I) live near the galactic plane (|b| < 5 deg)
   - LSST DDFs are high-galactic-latitude fields → no MW Cepheids expected
   - Cepheids in external galaxies (e.g. M49) are m~26-30 → below LSST 5σ
     alert threshold for a single epoch
   - Galactic SV fields (Carina, Trifid-Lagoon) are the correct hunting ground
   - If galactic fields have data: check above counts for 'galactic' field_type
   - If still zero: the LSST-Y1 galactic coverage may be minimal/absent yet

6. RECOMMENDATION:
   → Use f:xm_gcvs_type and f:xm_vsx_Type which have the most variable star coverage
   → Check if Carina / Trifid-Lagoon fields have been observed: see field counts above
   → Alternatively use the Fink `tags` endpoint with 'cataloged' to scan all fields
   → For external-galaxy Cepheids in M49: need stacking / forced photometry at known positions
""")
print("=" * 65)

## 14. Bonus: Period–Luminosity diagram for any pulsators found

In [ ]:
if df_periods.empty or df_periods["best_period_d"].isna().all():
    print("No period data available.")
else:
    # Add median r-band magnitude
    mag_med = []
    for oid in df_periods["diaObjectId"]:
        data = lc_dict.get(oid, {})
        if data and not data["src"].empty:
            df_r = filter_lc(data["src"])
            df_r = df_r[df_r["r:band"] == "r"]
            mag_med.append(np.nanmedian(df_r["mag"]) if len(df_r) > 0 else np.nan)
        else:
            mag_med.append(np.nan)
    df_periods["mag_r_median"] = mag_med

    df_pl = df_periods.dropna(subset=["best_period_d", "mag_r_median"])
    df_pl = df_pl[(df_pl["best_period_d"] > 0.1) & (df_pl["best_period_d"] < 200)]

    class_marker = {
        "cepheid_classical": ("*", "red", 100),
        "cepheid_type2": ("^", "darkorange", 80),
        "rr_lyrae": ("o", "dodgerblue", 40),
        "delta_scuti": ("s", "green", 30),
        "lpv_mira": ("D", "purple", 40),
        "rv_tauri": ("P", "brown", 50),
        "other_pulsator": ("x", "grey", 25),
    }

    fig, ax = plt.subplots(figsize=(9, 6))
    for cls, (marker, color, size) in class_marker.items():
        sub = df_pl[df_pl["pulsator_class"] == cls]
        if sub.empty:
            continue
        ax.scatter(
            np.log10(sub["best_period_d"]),
            sub["mag_r_median"],
            marker=marker,
            color=color,
            s=size,
            alpha=0.8,
            label=f"{cls} (N={len(sub)})",
            zorder=3,
        )

    # Leavitt law reference line (Cepheids, apparent only, rough)
    logP = np.linspace(0, 2, 50)
    # MW calibration: <M_V> = -2.81 log P - 1.43 (Feast & Catchpole 1997)
    # With DM ~ 11 (LMC, for reference) → apparent m ~ -2.81 logP + 9.57
    ax.plot(logP, -2.81 * logP + 11.5, "k--", lw=1, alpha=0.4, label="Leavitt law (rough, DM≈11)")

    ax.set_xlabel("log₁₀(Period / days)")
    ax.set_ylabel("Median r-band AB mag (apparent)")
    ax.invert_yaxis()
    ax.set_title(
        "Period–Luminosity diagram — Pulsating variables in LSST fields\n"
        "(apparent magnitudes, no distance or extinction correction)"
    )
    ax.legend(fontsize=8, loc="best")
    plt.tight_layout()
    savefig("pulsators_PL_diagram")
    plt.show()

## 15. Save light curves to disk

Persists all data produced by this notebook so that
`03_reload_cepheids_andanalyse.ipynb` can reload everything
without re-querying the Fink API.

### Layout inside `data_CEPHEIDS_DDF_02/`

```
data_CEPHEIDS_DDF_02/
├── df_obj.parquet            # all objects from cone searches
├── df_pulsators.parquet      # filtered pulsating-variable candidates
├── df_periods.parquet        # Lomb-Scargle period results
├── lc_dict_meta.parquet      # object-level metadata (one row / object)
└── lightcurves/
    └── {diaObjectId}/
        ├── sources.parquet   # DIA detections (r: columns)
        └── fp.parquet        # forced-photometry epochs
```


In [ ]:
# ── Section 15: save everything to disk ──────────────────────────────────────
import pathlib

LC_DIR = pathlib.Path(DIR_DATA) / "lightcurves"
LC_DIR.mkdir(parents=True, exist_ok=True)

# 1) Save summary DataFrames
if "df_obj" in dir() and not df_obj.empty:
    df_obj.to_parquet(pathlib.Path(DIR_DATA) / "df_obj.parquet", index=False)
    print(f"Saved df_obj          : {len(df_obj):,} rows")
else:
    print("WARNING: df_obj not found or empty — skipping")

if "df_pulsators" in dir() and not df_pulsators.empty:
    df_pulsators.to_parquet(pathlib.Path(DIR_DATA) / "df_pulsators.parquet", index=False)
    print(f"Saved df_pulsators    : {len(df_pulsators):,} rows")
else:
    print("WARNING: df_pulsators not found or empty — skipping")

if "df_periods" in dir() and not df_periods.empty:
    df_periods.to_parquet(pathlib.Path(DIR_DATA) / "df_periods.parquet", index=False)
    print(f"Saved df_periods      : {len(df_periods):,} rows")
else:
    print("WARNING: df_periods not found or empty — skipping")

# 2) Save per-object light curves + metadata
meta_rows = []

if "lc_dict" in dir() and lc_dict:
    for oid, data in lc_dict.items():
        obj_dir = LC_DIR / str(oid)
        obj_dir.mkdir(exist_ok=True)

        # Sources
        df_src = data.get("src", pd.DataFrame())
        if not df_src.empty:
            df_src.to_parquet(obj_dir / "sources.parquet", index=False)

        # Forced photometry
        df_fp = data.get("fp", pd.DataFrame())
        if not df_fp.empty:
            df_fp.to_parquet(obj_dir / "fp.parquet", index=False)

        # Collect metadata (scalar values only)
        meta = data.get("meta", {})
        row = {"diaObjectId": oid, "n_src": len(df_src), "n_fp": len(df_fp)}
        for k, v in meta.items():
            if isinstance(v, (str, int, float, bool, type(None))):
                row[k] = v
        meta_rows.append(row)

    df_meta = pd.DataFrame(meta_rows)
    df_meta.to_parquet(pathlib.Path(DIR_DATA) / "lc_dict_meta.parquet", index=False)
    print(f"Saved lc_dict_meta    : {len(df_meta):,} objects")
    print(f"Saved individual LCs  → {LC_DIR}")
else:
    print("WARNING: lc_dict not found or empty — no light curves saved")

print("\nAll data saved. Ready for 03_reload_cepheids_andanalyse.ipynb")